# Chapitre 2 — Calculer avec des vecteurs et des matrices · Corrigé

*Maths pour l'IA, pas à pas*, Belk Siri — notebook compagnon.

[Ouvrir dans Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/fr/ch02_corrige.ipynb) · Page compagnon : https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/

Chaque réponse commence par le raisonnement sur papier, puis montre son équivalent en Python.

**Règle de prédiction du chapitre** (donnée, et non ajustée) : poids $\mathbf{w} = (3 ; -8)$ en k€/m² et k€/km, ordonnée à l'origine $b = 45$ k€.

Avec les quartiers (leçon 2.4), la règle devient $\mathbf{w} = (3 ; -8 ; 12 ; -10)$, $b = 45$ k€, la Gare étant le quartier de référence.

**Sommaire.** Données · Les calculs des leçons · Exercices 2.1 à 2.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Couleurs du livre : une couleur = un rôle (observé, prédit, écart, droite du collègue)
BLUE, ORANGE, RESIDUAL, GREEN = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Données

Les identifiants (D2.1, D2.E2) sont ceux du livre et des fichiers CSV.

In [ ]:
# Données du chapitre 2, recopiées des fichiers CSV du livre (ne pas modifier)

# D2.1 — les sept logements avec leur distance au centre (leçons 2.1 à 2.3)
homes = pd.DataFrame({
    'home':    ['A', 'B', 'C', 'D', 'E', 'F', 'G'],
    'floor_area':  [32, 45, 54, 70, 85, 98, 62],
    'distance': [3.0, 1.5, 2.5, 1.0, 2.5, 0.5, 2.5],
    'price':   [118, 172, 185, 245, 279, 335, 209],
})

# D2.4 — cinq logements de l'est de la ville, avec leur quartier (leçon 2.4)
east = pd.DataFrame({
    'home':    ['K', 'L', 'M', 'N', 'O'],
    'district':    ['Parc', 'Rocade', 'Gare', 'Parc', 'Rocade'],
    'floor_area':  [60, 60, 60, 45, 80],
    'distance': [2.0, 2.0, 2.0, 1.0, 3.0],
    'price':   [223, 200, 210, 186, 248],
})

# D2.5 — les cinq T2 du Sud avec leur distance au centre (leçon 2.5)
t2_south = pd.DataFrame({
    'sale':       ['S03', 'S04', 'S05', 'S06', 'S07'],
    'floor_area':  [43, 40, 44, 46, 52],
    'distance': [3.5, 2.0, 1.5, 2.5, 0.5],
})

# D2.E2 — trois logements mis en vente (exercice 2.2)
new_homes = pd.DataFrame({
    'home':    ['H', 'I', 'J'],
    'floor_area':  [40, 75, 90],
    'distance': [0.5, 3.5, 1.0],
})

### Les variables des données

Les noms de code sont en anglais. Pour les colonnes, la dernière colonne donne le nom utilisé dans les fichiers CSV du livre.

| Nom | Sens | Unité | Colonne du CSV |
|:--|:--|:--|:--|
| `home` | identifiant du logement (A, B…) | — | `logement` |
| `floor_area` | surface habitable du logement | m² | `surface_m2` |
| `price` | prix de vente | k€ | `prix_kEUR` |
| `distance` | distance au centre-ville | km | `distance_km` |
| `district` | quartier du logement | — | `quartier` |
| `sale` | identifiant de la vente (S01…) | — | `vente` |

## Les calculs des leçons, en Python

Ces cellules refont les calculs des leçons 2.1 à 2.5. Exécutez-les et comparez avec le livre.

In [ ]:
# Leçon 2.1 — la règle de l'agence appliquée à G, terme par terme (équation 2.1)
area_part = 3 * 62        # k€/m² × m² = k€
distance_part = -8 * 2.5     # k€/km × km = k€
fixed_part = 45               # k€
price_G = area_part + distance_part + fixed_part
print(area_part, distance_part, fixed_part, '→', price_G, 'k€ ; écart signé :', 209 - price_G, 'k€')

In [ ]:
# Leçon 2.2 — vecteurs, produit scalaire (équation 2.2) et prédiction (équation 2.3)
x_G = np.array([62, 2.5])    # (surface en m², distance en km)
w = np.array([3, -8])        # (k€/m², k€/km), même ordre que x
b = 45                       # k€
print('dimension de x_G :', x_G.shape, '  w · x_G =', np.dot(w, x_G), '  ŷ =', np.dot(w, x_G) + b)
print('poids dans le mauvais ordre :', np.dot(w[::-1], x_G) + b, 'k€ (absurde)')

In [ ]:
# Leçon 2.3 — toutes les prédictions d'un coup (équation 2.5)
X = homes[['floor_area', 'distance']].to_numpy(dtype=float)
y = homes['price'].to_numpy(dtype=float)
y_hat = X @ w + b
y_chapeau = y_hat   # alias du livre (nom de l'édition française imprimée)
print('X :', X.shape, ' w :', w.shape, ' ŷ :', y_hat.shape)
print('ŷ =', y_hat)
print('écarts signés y − ŷ =', y - y_hat)
print('moyenne des écarts en valeur absolue, 7 logements :', round(np.mean(np.abs(y - y_hat)), 2), 'k€')
print('idem, A à F :', round(np.mean(np.abs(y - y_hat)[:6]), 2), 'k€ ; droite de l\'ouverture, A à F :',
      round(np.mean(np.abs(y - (3 * X[:, 0] + 30))[:6]), 2), 'k€')

In [ ]:
# Leçon 2.4 — colonnes indicatrices (équation 2.6), la Gare comme référence
dummies = pd.get_dummies(east['district'], drop_first=True).astype(int)   # colonnes Parc, Rocade
X_east = np.column_stack([east['floor_area'], east['distance'], dummies['Parc'], dummies['Rocade']])
w_east = np.array([3, -8, 12, -10])
y_east = east['price'].to_numpy(dtype=float)
print(dummies.columns.tolist(), ' X :', X_east.shape)
print('avec le quartier :', X_east @ w_east + 45)
print('sans le quartier :', X_east[:, :2] @ w_east[:2] + 45)
print('moyenne des écarts en valeur absolue : sans', np.mean(np.abs(y_east - (X_east[:, :2] @ w_east[:2] + 45))),
      ' avec', np.mean(np.abs(y_east - (X_east @ w_east + 45))))

In [ ]:
# Leçon 2.5 — la même prédiction dans deux échelles (équation 2.7), division par n
X5 = t2_south[['floor_area', 'distance']].to_numpy(dtype=float)
mu, sigma = X5.mean(axis=0), X5.std(axis=0, ddof=0)
w, b = np.array([3, -8]), 45
w_std, b_std = w * sigma, b + np.sum(w * mu)
Z = (X5 - mu) / sigma
print('moyennes :', mu, ' écarts-types :', sigma)
print('poids standardisés :', w_std, ' ordonnée :', b_std)
print('unités d\'origine :', X5 @ w + b)
print('échelle standardisée :', Z @ w_std + b_std)

## Exercice 2.1 — Lire un schéma

*Notions : produit d'une matrice par un vecteur (leçon 2.3).* Reprenez le schéma matriciel de la leçon 2.3 (les sept logements A à G). La cellule ci-dessous reconstruit $X$, $\mathbf{w}$ et $b$ pour vérifier vos réponses.

In [ ]:
X = homes[['floor_area', 'distance']].to_numpy(dtype=float)
w, b = np.array([3, -8]), 45
print(X)

In [ ]:
dimensions = {'X': (7, 2), 'w': (2,), 'y_hat': (7,)}
row_280 = 'E'
dimensions_c = {'X': (8, 3), 'w': (3,), 'y_hat': (8,)}

In [ ]:
# Vérification
y_hat = X @ w + b
y_chapeau = y_hat   # alias du livre (nom de l'édition française imprimée)
truth = {'X': X.shape, 'w': w.shape, 'y_hat': y_hat.shape}
if None in dimensions.values() or row_280 is None or None in dimensions_c.values():
    print('Complétez la cellule précédente.')
else:
    print('a) dimensions :', 'correct' if dimensions == truth else 'à revoir')
    letter = homes['home'][int(np.argmin(np.abs(y_hat - 280)))]
    print('b) logement :', 'correct' if row_280 == letter else 'à revoir')
    X8 = np.zeros((8, 3)); w3 = np.zeros(3)   # forme seulement : 8 logements, 3 caractéristiques
    print('c) dimensions :', 'correct' if dimensions_c == {'X': X8.shape, 'w': w3.shape,
          'y_hat': (X8 @ w3).shape} else 'à revoir')

### Corrigé de l'exercice 2.1

**a)** $X$ : **7 × 2** (7 logements, 2 caractéristiques). $\mathbf{w}$ : **2** composantes. $\hat{\mathbf{y}}$ : **7** prédictions.

**b)** C'est la ligne **E** : 85 m² et 2,5 km. Avec $\mathbf{w} = (3 ; -8)$ et $b = 45$ : 3 × 85 + (−8) × 2,5 + 45 = 255 − 20 + 45 = **280 k€**.

**c)** Avec l'étage et un huitième logement : $X$ devient **8 × 3**, $\mathbf{w}$ a **3** composantes (un poids pour l'étage), $\hat{\mathbf{y}}$ compte **8** prédictions. Si l'on ajoute seulement un logement, $\mathbf{w}$ garde ses **2** composantes : le nombre de poids dépend des caractéristiques, pas du nombre de logements. $X$ devient alors 8 × 2 et $\hat{\mathbf{y}}$ compte 8 prédictions.

*Éléments attendus :* les trois dimensions ; les cinq nombres utilisés pour E ; la distinction entre lignes (logements) et colonnes (caractéristiques).

## Exercice 2.2 — Trois nouveaux logements

*Données D2.E2. Notions : produit scalaire (leçon 2.2), produit d'une matrice par un vecteur (leçon 2.3).* Les questions a) à c) se font d'abord sur papier, avec le tableau de l'exercice 2.2 du livre.

In [ ]:
new_homes

### a) Écrire

$\mathbf{x}_H = (40 ; 0{,}5)$, $\mathbf{x}_I = (75 ; 3{,}5)$, $\mathbf{x}_J = (90 ; 1{,}0)$, avec la surface en m² puis la distance en km. La matrice rassemble ces vecteurs en lignes, dans le même ordre :

$$X = \begin{pmatrix} 40 & 0{,}5 \\ 75 & 3{,}5 \\ 90 & 1{,}0 \end{pmatrix}$$

Elle a 3 lignes (une par logement) et 2 colonnes (une par caractéristique) : ses dimensions sont **3 × 2**.

### b) Calculer à la main

| Logement | surface : 3 k€/m² × … | distance : −8 k€/km × … | part fixe | prix prédit |
|:--:|:--|:--|--:|--:|
| H | 3 × 40 = 120 k€ | −8 × 0,5 = −4 k€ | 45 k€ | **161 k€** |
| I | 3 × 75 = 225 k€ | −8 × 3,5 = −28 k€ | 45 k€ | **242 k€** |
| J | 3 × 90 = 270 k€ | −8 × 1,0 = −8 k€ | 45 k€ | **307 k€** |

Chaque terme est en k€ : le m² s'annule avec le « par m² », le km avec le « par km ».

In [ ]:
price_by_hand = {'H': 161, 'I': 242, 'J': 307}

In [ ]:
# Vérification (calcul indépendant, ligne par ligne)
for _, l in new_homes.iterrows():
    truth = 3 * l['floor_area'] - 8 * l['distance'] + 45
    yours = price_by_hand[l['home']]
    status = 'pas encore complété' if yours is None else ('correct' if np.isclose(yours, truth) else 'à revoir')
    print(f"{l['home']} : {status}")

### c) Interpréter

Droite de l'ouverture (3 × surface + 30) : H → 150 k€, I → 255 k€, J → 300 k€. Différences (nouvelle règle − droite) : H **+11**, I **−13**, J **+7** k€. La prédiction change le plus pour **I**, le logement le plus éloigné du centre (3,5 km). Elle baisse de 13 k€.

*Pour aller plus loin.* La différence entre les deux règles vaut $(3x_1 - 8x_2 + 45) - (3x_1 + 30) = 15 - 8x_2$ : elle ne dépend que de la distance. Elle est positive sous 15/8 ≈ 1,9 km et négative au-delà. H (0,5 km) gagne 15 − 4 = 11 k€, I (3,5 km) perd 28 − 15 = 13 k€, J (1 km) gagne 15 − 8 = 7 k€.

*Éléments attendus :* les trois prix de la droite ; les trois différences ; I identifié et relié à sa distance.

### d) Toutes les prédictions en une ligne

Complétez la cellule, puis exécutez la vérification.

In [ ]:
X = new_homes[['floor_area', 'distance']].to_numpy(dtype=float)   # 3 × 2
w = np.array([3, -8])    # k€/m², k€/km
b = 45                   # k€
y_hat = X @ w + b    # équation 2.5
y_chapeau = y_hat   # alias du livre (nom de l'édition française imprimée)

`X @ w` calcule les trois produits scalaires (une ligne de $X$ avec $\mathbf{w}$ à chaque fois). NumPy ajoute ensuite $b$ à chacun.

In [ ]:
# Vérification
if y_hat is None:
    print('Complétez la cellule précédente.')
else:
    print('dimensions :', np.shape(X), '@', np.shape(w), '→', np.shape(y_hat))
    print('dimensions :', 'correct' if np.shape(X) == (3, 2) and np.shape(y_hat) == (3,) else 'à revoir')
    print('prédictions :', y_hat, '→',
          'correct' if np.allclose(y_hat, [161, 242, 307]) else 'à revoir')

### e) Prédire, puis vérifier

**Prédiction.** Le poids −8 multiplie maintenant la surface (des dizaines de m²) et le poids 3 la distance (quelques km). Le terme −8 × surface l'emporte largement : les trois prix seront **négatifs**.

**Sur papier.** H : 3 × 0,5 − 8 × 40 + 45 = 1,5 − 320 + 45 = −273,5 k€. I : 3 × 3,5 − 8 × 75 + 45 = 10,5 − 600 + 45 = −544,5 k€. J : 3 × 1 − 8 × 90 + 45 = 3 − 720 + 45 = −672 k€.

**Pourquoi aucune erreur ?** NumPy vérifie seulement les dimensions : 3 × 2 multiplié par 2 donne 3. Elles sont toujours compatibles, et NumPy ne connaît ni le sens ni l'unité des colonnes. C'est à vous de garder le même ordre dans $X$ et dans $\mathbf{w}$. Un contrôle simple consiste à vérifier qu'un prix prédit est plausible, ou à garder les noms des colonnes avec pandas.

In [ ]:
if y_hat is not None:
    X_inverse = X[:, ::-1]          # colonnes échangées : distance, surface
    print('colonnes échangées :', X_inverse @ w + b)

## Exercice 2.3 — Que disent les poids ?

*Notions : lecture d'un poids (leçon 2.2), colonnes indicatrices (leçon 2.4).* Règle : $\mathbf{w} = (3 ; -8 ; 12 ; -10)$ pour surface, distance, Parc, Rocade ; $b = 45$ k€ ; la Gare est la référence. Questions sur papier, vérification ci-dessous.

In [ ]:
a_park_minus_ring_road = 12 - (-10)        # = 22 k€
b_far_minus_near = -8 * 1.5     # = −12 k€
c_far_park_minus_station = 12 - 8 * 1.5   # = 0 k€

In [ ]:
# Vérification : on calcule la règle pour des paires de logements fictifs
rule = lambda floor_area, distance, park, ring_road: 3 * floor_area - 8 * distance + 12 * park - 10 * ring_road + 45
truths = {'a': rule(60, 2, 1, 0) - rule(60, 2, 0, 1),
         'b': rule(60, 3.5, 0, 0) - rule(60, 2, 0, 0),
         'c': rule(60, 3.5, 1, 0) - rule(60, 2, 0, 0)}
for k, yours in [('a', a_park_minus_ring_road), ('b', b_far_minus_near), ('c', c_far_park_minus_station)]:
    status = 'pas encore complété' if yours is None else ('correct' if np.isclose(yours, truths[k]) else 'à revoir')
    print(f'{k}) {status}')

### Corrigé de l'exercice 2.3

**a)** Seuls les termes de quartier diffèrent : $12 \times 1 - (-10) \times 1 = 12 + 10 =$ **22 k€** de plus au Parc qu'à la Rocade.

**b)** Seul le terme de distance diffère : $-8 \text{ k€/km} \times 1{,}5 \text{ km} =$ **−12 k€**. L'appartement le plus éloigné est prédit 12 k€ moins cher.

**c)** Les deux effets se compensent : +12 k€ pour le Parc, −12 k€ pour les 1,5 km supplémentaires. La règle prédit **le même prix**. Chaque poids se lit « toutes choses égales par ailleurs ». Quand deux caractéristiques changent à la fois, on additionne leurs effets.

**d)** $b = 45$ k€ est le prix prédit pour un logement de 0 m², situé à 0 km du centre, dans le quartier de référence (la Gare). Un tel logement n'existe pas : $b$ fixe le niveau de la règle, comme les 30 k€ de l'ouverture.

*Éléments attendus :* les trois écarts avec leur unité ; l'idée de compensation ; la lecture « par rapport à la Gare » des poids de quartier ; $b$ sans réalité physique.

## Exercice 2.4 — Des colonnes indicatrices en Python

*Données D2.4. Notions : colonnes indicatrices (leçon 2.4), produit d'une matrice par un vecteur (leçon 2.3).*

In [ ]:
east

In [ ]:
park = (east['district'] == 'Parc').astype(int).to_numpy()
ring_road = (east['district'] == 'Rocade').astype(int).to_numpy()
X4 = np.column_stack([east['floor_area'], east['distance'], park, ring_road])
w4 = np.array([3, -8, 12, -10])
b4 = 45
y4 = X4 @ w4 + b4

**Variante.** `pd.get_dummies(east['district'], drop_first=True)` crée directement les colonnes Parc et Rocade, en retirant la Gare, première catégorie dans l'ordre alphabétique. Vérifiez toujours quelle catégorie a été retirée : c'est elle qui sert de référence pour lire les poids.

In [ ]:
# Vérification
if y4 is None:
    print('Complétez la cellule précédente.')
else:
    print('colonnes indicatrices :', 'correct' if list(park) == [1, 0, 0, 1, 0] and list(ring_road) == [0, 1, 0, 0, 1] else 'à revoir')
    print('X4 :', np.shape(X4), '→', 'correct' if np.shape(X4) == (5, 4) else 'à revoir')
    print('prédictions :', y4, '→', 'correct' if np.allclose(y4, [221, 199, 209, 184, 251]) else 'à revoir')

**c)** Moyenne des écarts en valeur absolue, avec et sans les colonnes de quartier.

In [ ]:
y_obs = east['price'].to_numpy(dtype=float)
mae_without = np.mean(np.abs(y_obs - (X4[:, :2] @ w4[:2] + b4)))
mae_with = np.mean(np.abs(y_obs - y4))

In [ ]:
if mae_without is None or mae_with is None:
    print('Complétez la cellule précédente.')
else:
    print(f'sans : {mae_without:.1f} k€, avec : {mae_with:.1f} k€ →',
          'correct' if np.isclose(mae_without, 10.2) and np.isclose(mae_with, 1.8) else 'à revoir')

### Corrigé de l'exercice 2.4

**a)** Parc : (1, 0, 0, 1, 0). Rocade : (0, 1, 0, 0, 1). M, à la Gare, a deux zéros.

**b)** $\hat{\mathbf{y}} = (221 ; 199 ; 209 ; 184 ; 251)$ k€. Par exemple pour N : 3 × 45 − 8 × 1 + 12 × 1 − 10 × 0 + 45 = 135 − 8 + 12 + 45 = 184 k€.

**c)** Sans le quartier, les écarts signés valent +14, −9, +1, +14, −13 : moyenne des valeurs absolues 51 / 5 = **10,2 k€**. Avec le quartier : +2, +1, +1, +2, −3, soit 9 / 5 = **1,8 k€**. Comme dans la leçon, ces nombres décrivent ces cinq ventes. Ils ne mesurent pas ce que donnerait la règle sur des ventes futures.

## Exercice 2.5 — Prédire, puis vérifier : changer d'unité

*Données D2.5. Notions : changement d'échelle (leçon 2.5, équation 2.7).* Notez vos prédictions pour a) et b) avant d'exécuter les cellules.

**Prédictions attendues.** a) Le poids de la distance devient −0,008 k€ par mètre. L'ordonnée à l'origine reste 45 k€, et les cinq prédictions ne changent pas. b) La moyenne passe de 2 km à 2 000 m, l'écart-type de 1 km à 1 000 m. Le poids standardisé reste $w'_2 = -0{,}008 \times 1\,000 = -8$ k€ par écart-type : il ne dépend pas de l'unité choisie.

In [ ]:
def standardized_weights(w, b, mu, sigma):
    w_std = w * sigma              # k€ par écart-type
    b_std = b + np.sum(w * mu)     # prix prédit du logement moyen
    return w_std, b_std
poids_standardises = standardized_weights   # alias du livre (nom de l'édition française imprimée)

In [ ]:
# Deux versions de la même règle : distance en km, puis en mètres
X_km = t2_south[['floor_area', 'distance']].to_numpy(dtype=float)
X_m = X_km * np.array([1, 1000])
w_km, w_m, b = np.array([3, -8]), np.array([3, -0.008]), 45
print('prédictions (km) :', X_km @ w_km + b)
print('prédictions (m)  :', X_m @ w_m + b)
results = {}
for name, Xv, wv in [('km', X_km, w_km), ('m', X_m, w_m)]:
    mu, sigma = Xv.mean(axis=0), Xv.std(axis=0, ddof=0)
    results[name] = standardized_weights(wv, b, mu, sigma)
    print(name, ': moyenne distance =', mu[1], ', écart-type =', sigma[1], ', poids standardisés =', results[name])

In [ ]:
# Vérification
if results['km'][0] is None:
    print('Complétez la fonction standardized_weights.')
else:
    ok = all(np.allclose(results[k][0], [12, -8]) and np.isclose(results[k][1], 164) for k in results)
    print('poids standardisés identiques en km et en m :', 'correct' if ok else 'à revoir')

In [ ]:
if results['km'][0] is not None:
    fig, axes = plt.subplots(1, 2, figsize=(8, 2.6))
    for ax, title, values in [
            (axes[0], 'Poids bruts du 2e terme', {'km': w_km[1], 'm': w_m[1]}),
            (axes[1], 'Poids standardisés du 2e terme', {k: results[k][0][1] for k in results})]:
        ax.bar(list(values), list(values.values()), color=[BLUE, ORANGE])
        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()

### Corrigé de l'exercice 2.5

**a)** Un mètre vaut 1/1 000 km : le poids devient −8 / 1 000 = **−0,008 k€/m**. Comme chaque distance est multipliée par 1 000, chaque produit $w_2 x_2$ reste identique : l'ordonnée à l'origine (45 k€) et les cinq prédictions (146, 149, 165, 163, 197 k€) **ne changent pas**.

**b)** Moyenne 2 000 m, écart-type 1 000 m. Avec l'équation 2.7 : $w'_2 = -0{,}008 \times 1\,000 = -8$ et $b' = 45 + 3 \times 45 - 0{,}008 \times 2\,000 = 45 + 135 - 16 = 164$ k€, **exactement** comme en km.

**c)** Sur le graphique de gauche, le poids brut passe de −8 à −0,008 : sa taille dépend de l'unité et ne permet aucune comparaison. Sur celui de droite, les poids standardisés coïncident : une hausse d'un écart-type de la distance correspond toujours à −8 k€ dans la prédiction, la surface restant fixe. Cette lecture décrit la règle sur ces données. Elle ne mesure ni une importance générale ni un effet causal.

*Éléments attendus :* prédictions inchangées ; poids brut divisé par 1 000 ; poids standardisé inchangé ; conclusion sur la comparaison des poids.

## Exercice 2.6 — Défi : une règle, plusieurs écritures

*Données D2.4. Notions : leçons 2.3, 2.4 et 2.5.* Trois colonnes indicatrices : Parc, Rocade, Gare.

In [ ]:
station = (east['district'] == 'Gare').astype(int).to_numpy()
park3 = (east['district'] == 'Parc').astype(int).to_numpy()
ring_road3 = (east['district'] == 'Rocade').astype(int).to_numpy()
X3 = np.column_stack([east['floor_area'], east['distance'], park3, ring_road3, station])
print('somme des trois indicatrices, ligne par ligne :', X3[:, 2:].sum(axis=1))
print('règle du collègue (b = 40) :', X3 @ np.array([3, -8, 17, -5, 5]) + 40)

In [ ]:
district_weight_b50 = [7, -15, -5]   # chaque poids de quartier diminue de 5 quand b augmente de 5

In [ ]:
# Vérification
if district_weight_b50 is None:
    print('Complétez la cellule précédente.')
else:
    y50 = X3 @ np.array([3, -8] + list(district_weight_b50)) + 50
    print('prédictions avec b = 50 :', y50, '→', 'correct' if np.allclose(y50, [221, 199, 209, 184, 251]) else 'à revoir')

### Corrigé de l'exercice 2.6

**a)** K : 180 − 16 + 17 + 40 = 221 ; L : 180 − 16 − 5 + 40 = 199 ; M : 180 − 16 + 5 + 40 = 209 ; N : 135 − 8 + 17 + 40 = 184 ; O : 240 − 24 − 5 + 40 = 251. Ce sont **exactement** les prédictions du tableau de la leçon 2.4.

**b)** Chaque logement a un et un seul 1 parmi les trois indicatrices. Ajouter une même quantité $c$ aux trois poids de quartier ajoute donc $c$ à toutes les prédictions, ce que l'on compense en retirant $c$ à $b$. Pour $b = 50$, $c = -5$ par rapport à la règle de la leçon (12, −10, 0, $b = 45$) : poids **(7 ; −15 ; −5)**. N'importe quel $c$ convient : il existe une infinité de règles équivalentes.

**c)** Dans l'équation 2.7, on change les *colonnes* (unités d'origine ou scores standardisés), et à chaque écriture des colonnes correspond une seule règle. C'est une traduction, sans ambiguïté. Avec trois indicatrices, les colonnes restent les mêmes et une infinité de poids donnent pourtant les mêmes prédictions. C'est cette seconde situation qui gêne l'ajustement du chapitre 3 : rien dans les données ne permet de choisir entre ces règles. Retirer une indicatrice, celle de la référence, supprime l'ambiguïté.

*Éléments attendus :* les cinq prédictions ; l'argument de la somme des indicatrices égale à 1 ; la distinction entre changement d'échelle (règle unique pour des colonnes données) et redondance (règle non unique).